# Volume 13, Chapter 2: Siegel's theorem (Python companion)

Exercises first, solutions at the end. We compute class numbers of $\mathbb Q(\sqrt{-p})$ in two ways and watch $\log h(-p)/\log p$ crawl towards $\frac12$ (Theorem 2.13), we go hunting for a Siegel zero and find none, and we turn to a computer science question that hangs on the zeros of $L$-functions: **can the Miller–Rabin primality test be made deterministic?** Requires `numpy`, `mpmath` and `sympy`.

In [ ]:
import math
from math import gcd
import numpy as np
import mpmath as mp
from sympy import primerange, legendre_symbol

## Exercises

**P2.1 (★) Class numbers, two ways.** For a prime $p\equiv3\pmod4$, $p>3$, $h(-p)$ is the number of reduced forms $ax^2+bxy+cy^2$ of discriminant $-p$: $|b|\le a\le c$, $\gcd(a,b,c)=1$, and $b\ge0$ when $|b|=a$ or $a=c$.

(a) Count reduced forms for all such $p<5000$, check the formula $h(-p)=\frac{T(p)}{2-\chi_p(2)}$ of Exercise 2.5 ($T(p)$ = residues minus non-residues in $]0,\frac p2[$), and list the $p$ with $h(-p)=1$.

(b) With the formula, compute $h(-p)$ for all $p<2\cdot10^5$. Print the mean, minimum and maximum of $\frac{\log h(-p)}{\log p}$ over $[10^3,10^4[$, $[10^4,10^5[$, $[10^5,2\cdot10^5[$. Why is the approach to $\frac12$ so slow?

(c) $L(1,\chi_p)=\frac{\pi h(-p)}{\sqrt p}$. List the record lows of $L(1,\chi_p)$, with $L(1,\chi_p)\log\log p$ next to them, and the largest values. Littlewood (1928) showed that, under the generalised Riemann hypothesis, $\frac1{\log\log p}\ll L(1,\chi_p)\ll\log\log p$. Do the numbers look like that, or like a Siegel zero?

**P2.2 (★★) Hunting for a Siegel zero.** A real zero $\beta$ of $L(s,\chi_p)$ close to $1$ would make $L(\sigma,\chi_p)$ small on $[\beta,1]$.

(a) Let $S(x)=\sum_{n\le x}\chi_p(n)$. Show by partial summation that $L(\sigma,\chi_p)=\sigma\int_1^\infty S(x)x^{-\sigma-1}\,dx$ for $\sigma>0$; deduce that if $S(x)\ge0$ for all $x$, then $L(\sigma,\chi_p)>0$ for every $\sigma>0$, so $L(s,\chi_p)$ has no real zero in $]0,1[$. For how many primes $p\equiv3\pmod4$ below $3000$ does this criterion apply? (Since $S$ has period $p$, it suffices to check $x<p$.)

(b) For the others, evaluate $L(\sigma,\chi_p)$ on the grid $\sigma=0.5,0.55,\dots,1$, with $\sum_{n<N}\chi_p(n)n^{-\sigma}+\bar S N^{-\sigma}$, where $N$ is a multiple of $p$ near $10^6$ and $\bar S$ is the mean of $S$ over a period. Check the method against `mpmath.dirichlet` for one $p$. Which $p$ gives the smallest value, and where?

**P2.3 (★★) CS application: deterministic Miller–Rabin.** Write $n-1=2^sd$ with $d$ odd. An integer $a$ is a *witness* for an odd composite $n$ if $a^d\not\equiv1$ and $a^{2^rd}\not\equiv-1\pmod n$ for $0\le r<s$; a prime has no witnesses. Miller (1976) showed that under the generalised Riemann hypothesis every odd composite $n$ has a witness $a=O(\log^2n)$, and Bach (1990) made it explicit: $a\le2\log^2n$. Testing these $a$ gives a deterministic polynomial-time primality test, if GRH holds.

(a) Find the strong pseudoprimes to base $2$ (odd composites with no witness $a=2$) below $10^6$, and the least witness of each. Compare with $2\log^2n$.

(b) Check that $n=3215031751$ is a strong pseudoprime to the bases $2,3,5,7$, find its least witness and compare with $2\log^2n$.

(c) The same mechanism governs the least quadratic non-residue $n_2(p)$: list the records of $n_2(p)$ for primes $p<10^6$, against $2\log^2p$ (Bach's bound for it under GRH). Why would a Siegel zero, or any zero of $L(s,\chi)$ off the critical line, threaten such bounds?

## Solutions

### P2.1
**How to find it.** Reduced forms: $a\le\sqrt{p/3}$ since $p=4ac-b^2\ge4a^2-a^2$; for each $a$ and $b$, $c=\frac{b^2+p}{4a}$ must be an integer. For $T(p)$ mark the squares $k^2\bmod p$ in a boolean array.

In [ ]:
def h_forms(D):
    count, a = 0, 1
    while 3*a*a <= D:
        for b in range(-a+1, a+1):
            if (b*b + D) % (4*a) == 0:
                c = (b*b + D) // (4*a)
                if c >= a and gcd(gcd(a, abs(b)), c) == 1 and not (b < 0 and a == c):
                    count += 1
        a += 1
    return count

def T_half(p):
    k = np.arange(1, (p+1)//2, dtype=np.int64)
    res = np.zeros(p, dtype=bool); res[(k*k) % p] = True
    half = res[1:(p+1)//2]
    return int(2*half.sum() - len(half))

small = [p for p in primerange(7, 5000) if p % 4 == 3]
print("formula holds for all", len(small), "primes:",
      all(T_half(p) == (2 - legendre_symbol(2, p))*h_forms(p) for p in small))
print("h(-p) = 1 for p =", [p for p in [3] + small if h_forms(p) == 1])

P = np.array([p for p in primerange(7, 200001) if p % 4 == 3])
H = np.array([T_half(p) // (2 - (1 if p % 8 == 7 else -1)) for p in P])
r = np.log(H)/np.log(P)
for lo, hi in [(1e3, 1e4), (1e4, 1e5), (1e5, 2e5)]:
    m = (P >= lo) & (P < hi)
    print(f"[{lo:.0e},{hi:.0e}[: mean {r[m].mean():.3f}, min {r[m].min():.3f}, max {r[m].max():.3f}")

L1 = math.pi*H/np.sqrt(P)
best = 10
for p, l in zip(P, L1):
    if l < best:
        best = l
        print(f"record low p={p}: L(1)={l:.4f}, L(1)*loglog p={l*math.log(math.log(p)):.3f}")
for j in np.argsort(-L1)[:3]:
    print(f"large: p={P[j]}, L(1)={L1[j]:.3f}, L(1)/loglog p={L1[j]/math.log(math.log(P[j])):.3f}")

**What it shows.** The formula agrees with the form count for all $338$ primes, and $h(-p)=1$ exactly for $p=3,7,11,19,43,67,163$: the list of Heegner, Baker and Stark, restricted to primes. The mean of $\frac{\log h}{\log p}$ is about $0.39$, $0.42$, $0.43$ on the three ranges, with minima between $0.20$ and $0.28$. The reason is $\frac{\log h(-p)}{\log p}=\frac12+\frac{\log(L(1,\chi_p)/\pi)}{\log p}$: the correction decays only like $\frac1{\log p}$, and with $L(1,\chi_p)$ between about $0.22$ and $5.5$ it lies between $-0.23$ and $+0.05$ at $p=10^5$, where $\log p\approx11.5$. The record lows of $L(1,\chi_p)$ are $p=163$ ($0.246$), then $90787$ ($0.240$) and $166147$ ($0.224$), with $L(1)\log\log p$ between $0.4$ and $0.8$; the largest values have $L(1)/\log\log p\approx2.2$. That is Littlewood's picture, not a Siegel zero: a zero at distance $\delta$ from $1$ would force $L(1,\chi_p)\le7\delta\log^2p$ (Corollary 2.9's proof), far below $0.2$ if $\delta$ were tiny.

### P2.2
**How to find it.** (a) $\sum_{n\le N}\chi(n)n^{-\sigma}=S(N)N^{-\sigma}+\sigma\int_1^NS(x)x^{-\sigma-1}dx$, and $S$ is bounded, so letting $N\to\infty$ gives the formula; if $S\ge0$ and $S(1)=1$, the integrand is $\ge0$ and positive on $[1,2[$. (b) Over a full period $\chi$ sums to $0$, so the tail $\sum_{n\ge N}\chi(n)n^{-\sigma}$, by partial summation, is $\sigma\int_N^\infty S(x)x^{-\sigma-1}dx\approx\bar SN^{-\sigma}$, with an error of order at most $p^2N^{-\sigma-1}$.

In [ ]:
def chi_arr(p):
    k = np.arange(1, (p+1)//2, dtype=np.int64)
    res = np.zeros(p, dtype=bool); res[(k*k) % p] = True
    c = np.where(res, 1, -1); c[0] = 0
    return c

def L_sigma(sig, p, N0=10**6):
    c = chi_arr(p); K = -(-N0 // p); N = K*p
    n = np.arange(1, N, dtype=np.float64); ch = np.tile(c, K)[1:N]
    return float((ch * n**-sig).sum() + np.cumsum(c).mean() * N**-sig)

mp.mp.dps = 15
print("check p=19, sigma=1/2:", L_sigma(0.5, 19), float(mp.dirichlet(0.5, [int(x) for x in chi_arr(19)])))

primes = [p for p in primerange(7, 3000) if p % 4 == 3]
others = [p for p in primes if np.cumsum(chi_arr(p)).min() < 0]
print(f"criterion S(x)>=0 proves no real zero for {len(primes)-len(others)} of {len(primes)} primes")
grid = np.linspace(0.5, 1, 11)
lows = sorted((min(L_sigma(s, p) for s in grid), p,
               grid[int(np.argmin([L_sigma(s, p) for s in grid]))]) for p in others)
for v, p, s in lows[:5]:
    print(f"p={p}: min L = {v:.4f} at sigma = {s:.2f}")

**What it shows.** The method agrees with `mpmath` to about fourteen digits. The criterion $S\ge0$ settles $97$ of the $217$ primes at once (it fails already for $p=19$, where $S(3)=-1$). For the remaining $120$ no value on the grid is negative; the smallest is $L(0.55,\chi_{163})\approx0.068$, then $p=2683$ and $p=2203$, all near the left end of the grid, well away from $\sigma=1$, where a Siegel zero would live. A grid is not a proof, but together with $L(1,\chi_p)\ge\frac\pi{\sqrt p}$ and a bound on $L'$ it could be made one (Exercise 2.1 is the same idea). Computations of this kind, pushed much further, are why nobody expects Siegel zeros: Chowla conjectured that $L(\sigma,\chi)\ne0$ for $0<\sigma<1$ and every real $\chi$.

### P2.3
**How to find it.** Sieve to $10^6$, run the strong test with base $2$ on the odd composites, then try $a=3,4,\dots$ on the survivors. For the non-residues, Euler's criterion $a^{(p-1)/2}\equiv-1$.

In [ ]:
def strong(n, a):
    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2; s += 1
    x = pow(a, d, n)
    if x in (1, n - 1):
        return True
    for _ in range(s - 1):
        x = x*x % n
        if x == n - 1:
            return True
    return False

def least_witness(n):
    a = 2
    while strong(n, a):
        a += 1
    return a

N = 10**6
sieve = np.ones(N + 1, bool); sieve[:2] = False
for i in range(2, int(N**0.5) + 1):
    if sieve[i]:
        sieve[i*i::i] = False
spsp2 = [n for n in range(9, N + 1, 2) if not sieve[n] and strong(n, 2)]
W = [least_witness(n) for n in spsp2]
print(len(spsp2), "strong pseudoprimes to base 2 below 1e6, first:", spsp2[:6])
print("largest least witness among them:", max(W), "; 2 log^2 n at n=2047:", round(2*math.log(2047)**2))

n = 3215031751
print("bases fooled among 2,3,5,7:", [a for a in (2, 3, 5, 7) if strong(n, a)],
      "; least witness:", least_witness(n), "; 2 log^2 n =", round(2*math.log(n)**2))

rec = 0
for p in range(3, N + 1, 2):
    if sieve[p]:
        a = 2
        while pow(a, (p - 1)//2, p) == 1:
            a += 1
        if a > rec:
            rec = a
            print(f"p={p}: least non-residue {a}, 2 log^2 p = {2*math.log(p)**2:.0f}")

**What it shows.** There are $46$ strong pseudoprimes to base $2$ below $10^6$, starting $2047=23\cdot89$, and every one of them is caught by $a=3$: below $10^6$ the least witness is never more than $3$, against $2\log^2n\ge116$. The number $3215031751=151\cdot751\cdot28351$, found by Pomerance, Selfridge and Wagstaff (1980), fools $2,3,5,7$; its least witness is $11$, against $2\log^2n\approx958$. The records of the least non-residue climb as $2,3,5,7,11,13,17,19,23,29,31,43$, the last at $p=366791$, against $2\log^2p\approx328$.

Why GRH? A composite $n$ without small witnesses has its small $a$ all inside a proper subgroup of $(\mathbb Z/n)^\times$, so some non-principal character $\chi$ (trivial on that subgroup) has $\chi(a)=1$ for all $a\le y$. Then $\sum_{a\le y}\chi(a)\Lambda(a)\approx\psi(y)\approx y$, while the explicit formula says this sum is $-\sum_\rho\frac{y^\rho}\rho+\dots$, a sum over the zeros of $L(s,\chi)$. If all zeros have real part $\frac12$, the sum is $O(\sqrt y\log^2(ny))$, a contradiction once $y\gg\log^2n$. A zero $\beta$ close to $1$, a Siegel zero above all, would let $y^\beta$ imitate $y$ and the argument collapses. Unconditionally, the best bound for the least non-residue is Burgess's $p^{1/(4\sqrt e)+\varepsilon}$ (1957), exponentially worse; and the first deterministic polynomial-time test that needs no hypothesis, by Agrawal, Kayal and Saxena (2002), is far slower in practice than Miller–Rabin.

*References.* G. L. Miller, "Riemann's hypothesis and tests for primality", *J. Comput. System Sci.* 13 (1976), 300–317. E. Bach, "Explicit bounds for primality testing and related problems", *Math. Comp.* 55 (1990), 355–380. C. Pomerance, J. L. Selfridge, S. S. Wagstaff, "The pseudoprimes to $25\cdot10^9$", *Math. Comp.* 35 (1980), 1003–1026. D. A. Burgess, "The distribution of quadratic residues and non-residues", *Mathematika* 4 (1957), 106–112. M. Agrawal, N. Kayal, N. Saxena, "PRIMES is in P", *Ann. of Math.* 160 (2004), 781–793. J. E. Littlewood, "On the class-number of the corpus $P(\sqrt{-k})$", *Proc. London Math. Soc.* (2) 27 (1928), 358–372.